In [ ]:
import json
import random
import re
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from PIL import Image
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path(".").resolve()
DATASET_PATH = ROOT / "dataset.jsonl"
IMAGE_SIZE = 224
DIM = 256
N_LAYERS = 4
MAX_TEXT_LEN = 128
BATCH_SIZE = 4
EPOCHS = 8
LR = 3e-4
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
print("device:", DEVICE)
print("dataset:", DATASET_PATH)

### LLM

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, dim, causal=False):
        super().__init__()
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.scale = dim ** 0.5
        self.causal = causal

    def forward(self, x):

        return None


class TransformerBlock(nn.Module):
    def __init__(self, dim, causal=False):
        super().__init__()


    def forward(self, x):

        return None


class MiniLLM(nn.Module):

    def __init__(self, vocab_size, dim, n_layers):
        super().__init__()


    def forward(self, x):

        return None

### ViT-энкодер



In [ ]:
class VisionEncoder(nn.Module):
    def __init__(self, out_dim):
        super().__init__()

    def forward(self, images):

        return None

## Простой токенизатор по словам



In [ ]:
class WordTokenizer:
    PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"
    SPECIAL = [PAD, BOS, EOS, UNK]

    def __init__(self, texts):
        tokens = []
        for t in texts:
            tokens.extend(self._tokenize(t))

        vocab_tokens = sorted(set(tokens))
        self.itos = self.SPECIAL + vocab_tokens
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.pad_id = self.stoi[self.PAD]
        self.bos_id = self.stoi[self.BOS]
        self.eos_id = self.stoi[self.EOS]
        self.unk_id = self.stoi[self.UNK]

    @staticmethod
    def _tokenize(text):
        return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

    def __len__(self):
        return len(self.itos)

    def encode(self, text, max_len):
        toks = self._tokenize(text)
        ids = [self.bos_id] + [self.stoi.get(tok, self.unk_id) for tok in toks] + [self.eos_id]
        ids = ids[:max_len]
        ids += [self.pad_id] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        pieces = []
        for i in ids:
            tok = self.itos[i]
            if tok == self.EOS:
                break
            if tok not in self.SPECIAL:
                pieces.append(tok)

        text = " ".join(pieces)
        text = re.sub(r"\s+([.,!?;:])", r"\1", text)
        return text

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_image_path(image_field: str) -> Path:
    p = Path(image_field.replace("\\", "/"))
    if p.is_absolute():
        return p
    return (ROOT / p).resolve()


class CaptionDataset(Dataset):
    def __init__(self, rows, tokenizer, transform):
        self.rows = []
        for row in rows:
            img_path = resolve_image_path(row["image"])
            if img_path.exists():
                self.rows.append({"image": img_path, "text": row["text"]})
        self.tokenizer = tokenizer
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        item = self.rows[idx]
        image = Image.open(item["image"]).convert("RGB")
        image = self.transform(image)
        text_ids = torch.tensor(
            self.tokenizer.encode(item["text"], MAX_TEXT_LEN), dtype=torch.long
        )
        return image, text_ids


transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

all_rows = load_jsonl(DATASET_PATH)
tokenizer = WordTokenizer([r["text"] for r in all_rows])
full_ds = CaptionDataset(all_rows, tokenizer, transform)

n_val = max(1, len(full_ds) // 10)
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(
    full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"samples: train={len(train_ds)}, val={len(val_ds)}, vocab={len(tokenizer)}")

## SmallVLM

In [ ]:
class SmallVLM(nn.Module):


    def __init__(self, vocab_size, dim, n_layers):
        super().__init__()
        self.vision_encoder = VisionEncoder(dim)      
        self.text_embedding = nn.Embedding(vocab_size, dim) 
        self.language_model = MiniLLM(vocab_size, dim, n_layers)

    def forward(self, images, text_ids):

        # визуальные токены
        vision_tokens = self.vision_encoder(images)   # (B, N_vis, D)
        n_vis = vision_tokens.size(1)

        text_input_ids = text_ids[:, :-1]
        text_tokens = self.text_embedding(text_input_ids)

        # склеиваем
        joint_tokens = torch.cat([vision_tokens, text_tokens], dim=1)

        all_logits = self.language_model(joint_tokens)  # (B, N_vis+T-1, vocab)

        # берем только ту часть, которая соответствует тексту
        text_logits = all_logits[:, n_vis:, :]
        return text_logits

    @torch.no_grad()
    def generate(self, image, tokenizer, max_new_tokens=80):

        self.eval()

        image = image.unsqueeze(0).to(DEVICE)

        vision_tokens = self.vision_encoder(image)

        # начало последовательности
        generated_ids = torch.tensor([[tokenizer.bos_id]], device=DEVICE)

        for _ in range(max_new_tokens):
            text_tokens = self.text_embedding(generated_ids)
            joint_tokens = torch.cat([vision_tokens, text_tokens], dim=1)

            logits = self.language_model(joint_tokens) 
            next_token_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)

            generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

            if next_token_id.item() == tokenizer.eos_id:
                break

        return tokenizer.decode(generated_ids[0].tolist())

## Обучение

In [ ]:
def compute_loss(logits, text_ids, pad_id):
    targets = text_ids[:, 1:].contiguous()
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1),
        ignore_index=pad_id,
    )


model = SmallVLM(len(tokenizer), DIM, N_LAYERS).to(DEVICE)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR
)


def run_epoch(loader, train=True):
    model.train(train)
    total_loss, n_batches = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, text_ids in tqdm(loader, leave=False):
            images = images.to(DEVICE)
            text_ids = text_ids.to(DEVICE)
            logits = model(images, text_ids)
            loss = compute_loss(logits, text_ids, tokenizer.pad_id)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()
            n_batches += 1
    return total_loss / max(n_batches, 1)


for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    print(f"epoch {epoch:02d} | train loss {train_loss:.4f} | val loss {val_loss:.4f}")

In [ ]:
sample_idx = 0
image, text_ids = full_ds[sample_idx]
gt_text = tokenizer.decode(text_ids.tolist())
pred_text = model.generate(image, tokenizer)

print("GT:", gt_text[:300])

print("Pred:", pred_text[:300])

In [ ]:
ckpt_path = "small_vlm.pt"
torch.save({
    "model": model.state_dict(),
    "tokenizer_itos": tokenizer.itos,
    "config": {"dim": DIM, "n_layers": N_LAYERS, "max_text_len": MAX_TEXT_LEN},
}, ckpt_path)
print("saved:", ckpt_path)